## Завантажимо датасет

In [ ]:
import pandas as pd
path = "./data/housing.csv"
df = pd.read_csv(path)
df.head()


## Попередня обробка

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Приберемо рядки з порожніми даними
processed_df = df.dropna()

# Ми хочемо передбачити y - ціну будинку                                                                                  
y = processed_df["median_house_value"]
# За допомогою всіх інших параметрів                                                                                                                                                                                                          
X = processed_df.drop(columns=["median_house_value"])

In [ ]:
print(processed_df.head())
le = LabelEncoder()

# Один з параметрів є текстом, а не числом - переведемо його в числовий вигляд
processed_df["ocean_proximity"] = le.fit_transform(processed_df["ocean_proximity"])

print(list(zip(le.classes_, range(len(le.classes_)))))   

print(processed_df.head())

X = processed_df.drop(columns=["median_house_value"])

In [ ]:
print(df[df['ocean_proximity'] == "NEAR BAY"].head(1))
print(df[df['ocean_proximity'] == "<1H OCEAN"].head(1))
print(df[df['ocean_proximity'] == "INLAND"].head(1))
print(df[df['ocean_proximity'] == "NEAR OCEAN"].head(1))
print(df[df['ocean_proximity'] == "ISLAND"].head(1))
df.head()

## Спробуємо використати моделі

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
naive_mod = KNeighborsRegressor()
# naive_mod.predict(X)

In [ ]:
from sklearn.linear_model import LinearRegression

mod = LinearRegression()
mod.fit(X, y)
print(y[:3])
mod.predict(X)[:3]

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

mod = KNeighborsRegressor()
mod.fit(X, y)
print(y[:3])
mod.predict(X)[:3]

## Подивимося на графіку, як справилася наша модель

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
import matplotlib.pylab as plt

mod = KNeighborsRegressor().fit(X, y)
pred = mod.predict(X)
plt.scatter(pred, y)

## Обєднаємо все в пайплайн

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pylab as plt

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsRegressor())
])
pred = pipe.fit(X, y).predict(X)
plt.scatter(pred, y)

## Змінимо параметри моделі

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pylab as plt


# При кількості сусідів = 1, для передбачення точки ми просто дивимося на її саму
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=1))
])
pred = pipe.fit(X, y).predict(X)
plt.scatter(pred, y)

## Використаємо grid search для пошуку найоптимальніших параметрів моделі

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pandas as pd

mod = GridSearchCV(estimator=pipe,
                 param_grid={
                   'model__n_neighbors': [1, 3, 5, 10, 20, 50],                                                                           
                   'model__p': [1, 2],  # Manhattan vs Euclidean
                 },
                 cv=5)
mod.fit(X, y)

In [ ]:
pd.DataFrame(mod.cv_results_)

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=10, p=1))
])
pred = pipe.fit(X, y).predict(X)
plt.scatter(pred, y)